**1. Install & Import Dependencies**

In [ ]:
!pip install matplotlib seaborn torch torchvision tqdm scikit-learn

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

**2. Dataset Setup**

In [ ]:
data_dir = "/content/drive/MyDrive/sample"
images_dir = os.path.join(data_dir, "images")
csv_path = os.path.join(data_dir, "sample_labels.csv")

**3. Read Labels**

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImageWidth,OriginalImageHeight,OriginalImagePixelSpacing_x,OriginalImagePixelSpacing_y
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,5,13,060Y,M,AP,3056,2544,0.139,0.139
1,00000013_026.png,Cardiomegaly|Emphysema,26,13,057Y,M,AP,2500,2048,0.168,0.168
2,00000017_001.png,No Finding,1,17,077Y,M,AP,2500,2048,0.168,0.168
3,00000030_001.png,Atelectasis,1,30,079Y,M,PA,2992,2991,0.143,0.143
4,00000032_001.png,Cardiomegaly|Edema|Effusion,1,32,055Y,F,AP,2500,2048,0.168,0.168


In [ ]:
df["FirstLabel"] = df["Finding Labels"].apply(lambda x: x.split("|")[0])
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImageWidth,OriginalImageHeight,OriginalImagePixelSpacing_x,OriginalImagePixelSpacing_y,FirstLabel
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,5,13,060Y,M,AP,3056,2544,0.139,0.139,Emphysema
1,00000013_026.png,Cardiomegaly|Emphysema,26,13,057Y,M,AP,2500,2048,0.168,0.168,Cardiomegaly
2,00000017_001.png,No Finding,1,17,077Y,M,AP,2500,2048,0.168,0.168,No Finding
3,00000030_001.png,Atelectasis,1,30,079Y,M,PA,2992,2991,0.143,0.143,Atelectasis
4,00000032_001.png,Cardiomegaly|Edema|Effusion,1,32,055Y,F,AP,2500,2048,0.168,0.168,Cardiomegaly


In [ ]:
classes = sorted(df["FirstLabel"].unique())
print("Classes:", classes)

Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


**4. Train/Validation Split**

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["FirstLabel"])

print(len(train_df), len(val_df))

4484 1122


**5. Custom PyTorch Dataset**

In [ ]:
from PIL import Image

class ChestXrayDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["Image Index"])
        image = Image.open(img_path).convert("RGB")

        label = classes.index(row["FirstLabel"])

        if self.transform:
            image = self.transform(image)

        return image, label

**6. Transforms**

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

**7. Dataloaders**

In [ ]:
batch_size = 32

train_dataset = ChestXrayDataset(train_df, images_dir, train_transform)
val_dataset = ChestXrayDataset(val_df, images_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

**8. MODEL 1: Custom CNN**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256), nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model_cnn = SimpleCNN(num_classes=len(classes))

**9. MODEL 2: Transfer Learning (DenseNet-121)**

In [ ]:
model_tl = models.densenet121(weights="IMAGENET1K_V1")

# Freeze feature layers
for param in model_tl.features.parameters():
    param.requires_grad = False

# Replace classifier
num_features = model_tl.classifier.in_features
model_tl.classifier = nn.Linear(num_features, len(classes))

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 42.9MB/s]


**Pick the model you want to train:**

In [32]:
model = model_cnn  # model_tl or model_cnn

**10. Training Setup**

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

**1. Training Loop**

In [34]:
def train_model(model, train_loader, val_loader, epochs=5):
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            output = model(imgs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_losses.append(total_loss / len(train_loader))


        # Validation
        model.eval()
        val_loss = 0
        correct = 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                output = model(imgs)
                val_loss += criterion(output, labels).item()
                pred = output.argmax(dim=1)
                correct += (pred == labels).sum().item()

        val_losses.append(val_loss / len(val_loader))
        accuracy = correct / len(val_loader.dataset)

        print(f"Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f} | Val Loss={val_losses[-1]:.4f} | Val Acc={accuracy:.4f}")

    return train_losses, val_losses

**Run training:**

In [35]:
import re

try:
    train_losses, val_losses = train_model(model, train_loader, val_loader, epochs=5)
except FileNotFoundError as e:
    print(f"\nERROR: An image file was not found during training: {e}")
    print("Please check the following:")
    print(f"  1. Is your Google Drive mounted correctly?")
    print(f"  2. Is the 'images_dir' path correct: '{images_dir}'?")
    print("  3. Do all image files referenced in your dataset (especially the one mentioned in the error) exist in that directory?")

    # Attempt to extract the specific missing file from the error message for more specific guidance
    match = re.search(r"No such file or directory: '(.+)'", str(e))
    if match:
        problem_file = match.group(1)
        print(f"\nSpecifically, the file '{problem_file}' could not be found.")
        print("Please ensure this file is present and accessible.")

    train_losses = [] # Initialize to empty lists to avoid NameError if training didn't complete
    val_losses = []


Epoch 1/5: 100%|██████████| 141/141 [15:18<00:00,  6.52s/it]


Epoch 1: Train Loss=1.7235 | Val Loss=1.7433 | Val Acc=0.5428


Epoch 2/5:  35%|███▌      | 50/141 [04:50<08:48,  5.81s/it]


KeyboardInterrupt: 

**12. Plot Loss Curves**

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

**13. Confusion Matrix**

In [ ]:
y_true, y_pred = [], []
model.eval()

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        output = model(imgs)
        preds = output.argmax(dim=1).cpu()

        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, target_names=classes))